In [1]:
%pip install treys

Note: you may need to restart the kernel to use updated packages.


In [2]:
from treys import Card, Deck, Evaluator
import random

# Inicializar el evaluador oficial
evaluator = Evaluator()

def calcular_equity_monte_carlo(mis_cartas_str, mesa_str=[], num_rivales=1, simulaciones=2000):
    """
    Calcula la probabilidad matemática de victoria (Equity) usando simulación Monte Carlo.
    - mis_cartas_str: ej. ['Ah', 'Kd'] (As de corazones, Rey de diamantes)
    - mesa_str: ej. ['Qd', '9c', '5c'] (El Flop)
    """
    # 1. Convertir texto a cartas binarias de Treys
    mis_cartas = [Card.new(c) for c in mis_cartas_str]
    mesa = [Card.new(c) for c in mesa_str]
    
    victorias = 0
    empates = 0
    derrotas = 0
    
    cartas_conocidas = mis_cartas + mesa
    cartas_restantes_mesa = 5 - len(mesa)  # Cuántas cartas faltan por salir (Turn y River)
    
    for _ in range(simulaciones):
        deck = Deck()
        # Quitar las cartas que ya conocemos de la baraja
        for c in cartas_conocidas:
            deck.cards.remove(c)
            
        # Repartir cartas aleatorias a los rivales
        manos_rivales = []
        for _ in range(num_rivales):
            manos_rivales.append([deck.draw(1)[0], deck.draw(1)[0]])
            
        # Si faltan cartas en la mesa (ej. Turn o River), sacarlas de la baraja
        mesa_simulada = list(mesa)
        if cartas_restantes_mesa > 0:
            mesa_simulada.extend(deck.draw(cartas_restantes_mesa))
            
        # Evaluar la puntuación de mi mano (entre menor el número en Treys, más fuerte es la mano)
        mi_score = evaluator.evaluate(mesa_simulada, mis_cartas)
        
        # Evaluar a los rivales
        scores_rivales = [evaluator.evaluate(mesa_simulada, mano_r) for mano_r in manos_rivales]
        mejor_score_rival = min(scores_rivales)
        
        if mi_score < mejor_score_rival:
            victorias += 1
        elif mi_score == mejor_score_rival:
            empates += 1
        else:
            derrotas += 1
            
    equity = (victorias + (empates / 2.0)) / simulaciones
    return {
        "victorias_pct": round((victorias / simulaciones) * 100, 2),
        "empates_pct": round((empates / simulaciones) * 100, 2),
        "derrotas_pct": round((derrotas / simulaciones) * 100, 2),
        "equity_total": round(equity * 100, 2)
    }

# --- PRUEBA EN VIVO DEL MOTOR MATEMÁTICO ---
# Situación: Tienes As de tréboles y Rey de tréboles (Ac, Kc).
# En la mesa sale un Flop con dos tréboles: Reina de tréboles, 9 de tréboles y 2 de picas.
mis_cartas = ['Ac', 'Kc']
mesa = ['Qc', '9c', '2s']

print("🃏 EVALUANDO ESCENARIO DE PÓKER:")
print(f" • Mis cartas: {mis_cartas}")
print(f" • Mesa (Flop): {mesa} (¡Proyecto de Color con As-Rey!)")

print("\n⚙️ Ejecutando 2.000 simulaciones Monte Carlo en microsegundos...")
resultado = calcular_equity_monte_carlo(mis_cartas, mesa, num_rivales=1, simulaciones=2000)

print("=" * 60)
print("  REPORTE DEL MOTOR MATEMÁTICO DE EQUITY")
print("=" * 60)
print(f" • Probabilidad de ganar:  {resultado['victorias_pct']}%")
print(f" • Probabilidad de empatar: {resultado['empates_pct']}%")
print(f" • Probabilidad de perder:  {resultado['derrotas_pct']}%")
print(f" 🎯 EQUITY TOTAL:           {resultado['equity_total']}% frente a un rival")
print("=" * 60)

🃏 EVALUANDO ESCENARIO DE PÓKER:
 • Mis cartas: ['Ac', 'Kc']
 • Mesa (Flop): ['Qc', '9c', '2s'] (¡Proyecto de Color con As-Rey!)

⚙️ Ejecutando 2.000 simulaciones Monte Carlo en microsegundos...
  REPORTE DEL MOTOR MATEMÁTICO DE EQUITY
 • Probabilidad de ganar:  68.65%
 • Probabilidad de empatar: 1.5%
 • Probabilidad de perder:  29.85%
 🎯 EQUITY TOTAL:           69.4% frente a un rival


In [3]:
import numpy as np
import pandas as pd
import joblib
from tensorflow import keras

# 1. Cargar la red neuronal y el escalador que guardamos en el Paso 4
print("🧠 Cargando la Red Neuronal entrenada y el escalador...")
modelo_ia = keras.models.load_model("../models/v2_corregido/pht_t_modelo_poker.keras")
scaler_ia = joblib.load("../models/v2_corregido/scaler_poker.pkl")

def entrenador_inteligente(mis_cartas, mesa, bote, to_call, stack, calle_num=1):
    """
    Combina el Motor Matemático (Equity) con la Red Neuronal (Comportamiento Humano).
    """
    calle_nombres = {0: 'Preflop', 1: 'Flop', 2: 'Turn', 3: 'River'}
    
    # A. CÁLCULO MATEMÁTICO (EQUITY & POT ODDS)
    equity_info = calcular_equity_monte_carlo(mis_cartas, mesa, num_rivales=1, simulaciones=2000)
    pot_odds = to_call / (bote + to_call) if (bote + to_call) > 0 else 0.0
    equity_minima = pot_odds * 100
    
    # B. PREDICCIÓN NEURONAL (PATRÓN HUMANO)
    # Creamos un DataFrame con los nombres exactos de columnas para evitar warnings
    df_situacion = pd.DataFrame([{
        'calle': calle_num,
        'bote': float(bote),
        'to_call': float(to_call),
        'stack': float(stack),
        'pot_odds': float(pot_odds)
    }])
    
    situacion_escalada = scaler_ia.transform(df_situacion)
    prob_acciones = modelo_ia.predict(situacion_escalada, verbose=0)[0]
    
    # C. REPORTE COMPLETO 360°
    print("=" * 70)
    print("  ANÁLISIS COMPLETO DEL ENTRENADOR INTELIGENTE (PHT-T)")
    print("=" * 70)
    print(f"📍 ESCENARIO: {calle_nombres[calle_num]} | Bote: {bote} fichas | Apuesta a pagar: {to_call} | Stack: {stack}")
    print(f"🃏 TUS CARTAS: {mis_cartas} | MESA: {mesa}")
    
    print("\n--- 1. ANÁLISIS MATEMÁTICO OBJETIVO (Motor Combinatorio) ---")
    print(f" • Tu probabilidad de victoria (Equity):   {equity_info['equity_total']}%")
    print(f" • Equity mínima matemática para pagar:   {equity_minima:.2f}% (Pot Odds)")
    if equity_info['equity_total'] >= equity_minima:
        print("  💡 Veredicto Matemático: RENTABLE PAGAR O SUBIR (Tu equity supera el precio de la apuesta).")
    else:
        print("  ⚠️ Veredicto Matemático: NO RENTABLE (El precio de la apuesta supera tu probabilidad de ganar).")
        
    print("\n--- 2. PATRÓN HISTÓRICO HUMANO (Red Neuronal - Modelo 2) ---")
    print(f" • Frecuencia histórica de Retirarse (Fold):       {prob_acciones[0] * 100:.2f}%")
    print(f" • Frecuencia histórica de Igualar/Pasar (Call):   {prob_acciones[1] * 100:.2f}%")
    print(f" • Frecuencia histórica de Apostar/Subir (Raise):  {prob_acciones[2] * 100:.2f}%")
    print("=" * 70)

# Probemos el escenario de Flop con proyecto de color As-Rey
entrenador_inteligente(
    mis_cartas=['Ac', 'Kc'],
    mesa=['Qc', '9c', '2s'],
    bote=100.0,
    to_call=50.0,
    stack=1200.0,
    calle_num=1 # Flop
)

🧠 Cargando la Red Neuronal entrenada y el escalador...


  ANÁLISIS COMPLETO DEL ENTRENADOR INTELIGENTE (PHT-T)
📍 ESCENARIO: Flop | Bote: 100.0 fichas | Apuesta a pagar: 50.0 | Stack: 1200.0
🃏 TUS CARTAS: ['Ac', 'Kc'] | MESA: ['Qc', '9c', '2s']

--- 1. ANÁLISIS MATEMÁTICO OBJETIVO (Motor Combinatorio) ---
 • Tu probabilidad de victoria (Equity):   72.28%
 • Equity mínima matemática para pagar:   33.33% (Pot Odds)
  💡 Veredicto Matemático: RENTABLE PAGAR O SUBIR (Tu equity supera el precio de la apuesta).

--- 2. PATRÓN HISTÓRICO HUMANO (Red Neuronal - Modelo 2) ---
 • Frecuencia histórica de Retirarse (Fold):       54.19%
 • Frecuencia histórica de Igualar/Pasar (Call):   34.54%
 • Frecuencia histórica de Apostar/Subir (Raise):  11.27%


In [4]:
# Cargar df_train en este cuaderno para ver las estadísticas reales
train_path = "../data/processed/train.parquet"
df_train = pd.read_parquet(train_path)

# 1. Mirar la escala real de las partidas en el conjunto de entrenamiento
print("📊 PROMEDIOS Y MEDIANAS REALES EN NUESTRO DATASET:")
print(df_train[['bote', 'to_call', 'stack', 'pot_odds']].describe().round(2).T[['mean', 'std', 'min', '50%', 'max']])

print("\n" + "=" * 70)
print("🧪 PROBANDO 3 ESCENARIOS CON LA ESCALA REAL DEL PÓKER")
print("=" * 70)

# Escenario 1: Nadie ha apostado (to_call = 0) -> Debería ser Check o Bet, NUNCA Fold
print("\n🟢 CASO 1: Nadie ha apostado (to_call = 0.0, Gratis jugar)")
entrenador_inteligente(
    mis_cartas=['Ac', 'Kc'],
    mesa=['Qc', '9c', '2s'],
    bote=20.0,
    to_call=0.0,
    stack=200.0,
    calle_num=1
)

# Escenario 2: Apuesta moderada estándar (to_call = 6 en bote de 18)
print("\n🟡 CASO 2: Apuesta moderada estándar (to_call = 6.0 en bote de 18.0)")
entrenador_inteligente(
    mis_cartas=['Ac', 'Kc'],
    mesa=['Qc', '9c', '2s'],
    bote=18.0,
    to_call=6.0,
    stack=200.0,
    calle_num=1
)

# Escenario 3: Apuesta agresiva (to_call = 40 en bote de 30)
print("\n🔴 CASO 3: Apuesta agresiva (to_call = 40.0 en bote de 30.0)")
entrenador_inteligente(
    mis_cartas=['Ac', 'Kc'],
    mesa=['Qc', '9c', '2s'],
    bote=30.0,
    to_call=40.0,
    stack=200.0,
    calle_num=1
)

📊 PROMEDIOS Y MEDIANAS REALES EN NUESTRO DATASET:
             mean      std    min      50%       max
bote        97.78   211.10  10.00    45.00  14531.93
to_call     30.64   106.16   0.00    10.00  12401.93
stack     1972.19  1344.89   3.35  1839.00  14682.93
pot_odds     0.25     0.17   0.00     0.33      0.50

🧪 PROBANDO 3 ESCENARIOS CON LA ESCALA REAL DEL PÓKER

🟢 CASO 1: Nadie ha apostado (to_call = 0.0, Gratis jugar)



  ANÁLISIS COMPLETO DEL ENTRENADOR INTELIGENTE (PHT-T)
📍 ESCENARIO: Flop | Bote: 20.0 fichas | Apuesta a pagar: 0.0 | Stack: 200.0
🃏 TUS CARTAS: ['Ac', 'Kc'] | MESA: ['Qc', '9c', '2s']

--- 1. ANÁLISIS MATEMÁTICO OBJETIVO (Motor Combinatorio) ---
 • Tu probabilidad de victoria (Equity):   71.6%
 • Equity mínima matemática para pagar:   0.00% (Pot Odds)
  💡 Veredicto Matemático: RENTABLE PAGAR O SUBIR (Tu equity supera el precio de la apuesta).

--- 2. PATRÓN HISTÓRICO HUMANO (Red Neuronal - Modelo 2) ---
 • Frecuencia histórica de Retirarse (Fold):       0.05%
 • Frecuencia histórica de Igualar/Pasar (Call):   66.05%
 • Frecuencia histórica de Apostar/Subir (Raise):  33.89%

🟡 CASO 2: Apuesta moderada estándar (to_call = 6.0 en bote de 18.0)
  ANÁLISIS COMPLETO DEL ENTRENADOR INTELIGENTE (PHT-T)
📍 ESCENARIO: Flop | Bote: 18.0 fichas | Apuesta a pagar: 6.0 | Stack: 200.0
🃏 TUS CARTAS: ['Ac', 'Kc'] | MESA: ['Qc', '9c', '2s']

--- 1. ANÁLISIS MATEMÁTICO OBJETIVO (Motor Combinatorio) ---


  ANÁLISIS COMPLETO DEL ENTRENADOR INTELIGENTE (PHT-T)
📍 ESCENARIO: Flop | Bote: 30.0 fichas | Apuesta a pagar: 40.0 | Stack: 200.0
🃏 TUS CARTAS: ['Ac', 'Kc'] | MESA: ['Qc', '9c', '2s']

--- 1. ANÁLISIS MATEMÁTICO OBJETIVO (Motor Combinatorio) ---
 • Tu probabilidad de victoria (Equity):   73.72%
 • Equity mínima matemática para pagar:   57.14% (Pot Odds)
  💡 Veredicto Matemático: RENTABLE PAGAR O SUBIR (Tu equity supera el precio de la apuesta).

--- 2. PATRÓN HISTÓRICO HUMANO (Red Neuronal - Modelo 2) ---
 • Frecuencia histórica de Retirarse (Fold):       44.99%
 • Frecuencia histórica de Igualar/Pasar (Call):   41.10%
 • Frecuencia histórica de Apostar/Subir (Raise):  13.90%
